In [3]:
# ============================================================
# 📦 INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# Instalamos las bibliotecas necesarias:
# - transformers: maneja el modelo de lenguaje.
# - accelerate: optimización en GPU/CPU.
# - bitsandbytes: carga eficiente en 4 bits.
# - sentencepiece: requerido por algunos modelos.
# - arxiv: cliente oficial del API de arXiv.
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.0" "bitsandbytes>=0.44.0" sentencepiece arxiv


In [4]:
# ============================================================
# 🧩 IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import re                # Expresiones regulares (procesamiento de texto)
import torch             # Backend de PyTorch para ejecutar el modelo
import arxiv             # Cliente oficial del API de arXiv
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# AutoTokenizer: convierte texto ↔ tokens
# AutoModelForCausalLM: carga el modelo de lenguaje causal
# BitsAndBytesConfig: configura la carga del modelo en 4 bits


In [5]:
# ============================================================
# ⚙️ CONFIGURACIÓN DEL MODELO QWEN (OPEN SOURCE)
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # Modelo open source preentrenado
USE_4BIT = True                            # Activa cuantización en 4 bits para reducir VRAM
MAX_TOKENS = 1024                          # Tokens máximos de salida del modelo
TEMPERATURE = 0.2                          # Controla la creatividad (bajo = más estable)
device = "cuda" if torch.cuda.is_available() else "cpu"  # Detecta GPU si está disponible

print(f"🧠 Cargando modelo {MODEL_NAME} en {device}...")

# Configuramos la carga 4-bit moderna con BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Carga del modelo en 4 bits
    bnb_4bit_use_double_quant=True,       # Usa doble cuantización para mayor precisión
    bnb_4bit_quant_type="nf4",            # Tipo NF4 (cuantización eficiente)
    bnb_4bit_compute_dtype=torch.bfloat16 if device == "cuda" else torch.float16,
) if USE_4BIT else None

# Carga del tokenizador y del modelo
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None,  # Coloca capas automáticamente en GPU
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    quantization_config=bnb_config,                   # Configuración moderna
)

if device == "cpu":
    print("⚠️ Ejecutando en CPU — puede tardar más en responder.")

print("✅ Modelo cargado correctamente.")


🧠 Cargando modelo Qwen/Qwen2.5-7B-Instruct en cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Modelo cargado correctamente.


In [6]:
# ============================================================
# 🔎 FUNCIONES PARA CONSULTAR ARXIV
# ============================================================

def buscar_articulos_arxiv(consulta: str, max_resultados: int = 5):
    """
    Busca artículos en arXiv según una consulta textual.
    Ejemplo: 'diffusion models AND cat:cs.LG'
    """
    print(f"\n🔍 Buscando en arXiv: {consulta}")
    search = arxiv.Search(
        query=consulta,
        max_results=max_resultados,
        sort_by=arxiv.SortCriterion.LastUpdatedDate,
    )

    resultados = []
    for r in arxiv.Client().results(search):
        autores = ", ".join([a.name for a in r.authors[:6]])
        resumen = re.sub(r"\s+", " ", r.summary.strip())
        resultados.append(
            f"🧾 ID: {r.get_short_id()}\n"
            f"Título: {r.title.strip()}\n"
            f"Autores: {autores}\n"
            f"Publicado: {r.published.date()}\n"
            f"URL: {r.entry_id}\n"
            f"Resumen: {resumen}\n"
        )
    if not resultados:
        print("❌ No se encontraron resultados.")
    else:
        print("\n".join(resultados))


def detalle_articulo_arxiv(id_o_url: str):
    """
    Obtiene el detalle de un artículo de arXiv por ID o URL.
    Ejemplo: 'https://arxiv.org/abs/2401.01234' o '2401.01234'
    """
    print(f"\n📘 Consultando detalle de: {id_o_url}")
    m = re.search(r"arxiv\\.org/(abs|pdf)/([0-9]+\\.[0-9]+)", id_o_url)
    paper_id = m.group(2) if m else id_o_url.strip()

    search = arxiv.Search(id_list=[paper_id])
    resultados = list(arxiv.Client().results(search))

    if not resultados:
        print("❌ No se encontró el artículo.")
        return

    r = resultados[0]
    autores = ", ".join([a.name for a in r.authors])
    resumen = re.sub(r"\s+", " ", r.summary.strip())
    print(
        f"\n📄 Detalle del artículo {paper_id}\n"
        f"Título: {r.title.strip()}\n"
        f"Autores: {autores}\n"
        f"Publicado: {r.published.date()}\n"
        f"URL: {r.entry_id}\n"
        f"Resumen: {resumen}\n"
    )


In [7]:
# ============================================================
# 💬 FUNCIÓN PARA GENERAR RESPUESTAS CON EL MODELO QWEN
# ============================================================

def generar_respuesta(prompt: str):
    """
    Envía un texto (prompt) al modelo Qwen para obtener una respuesta.
    Ideal para resumir, analizar o interpretar artículos de arXiv.
    """
    print(f"\n🗣️ Enviando solicitud al modelo...\nPrompt: {prompt}\n")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    respuesta = tokenizer.decode(output[0], skip_special_tokens=True)

    print("\n🧩 Respuesta del modelo:\n")
    print(respuesta)
    return respuesta


In [12]:
# ============================================================
# 🚀 DEMOSTRACIÓN Y EJEMPLOS DE USO
# ============================================================

print("\n✅ Agente arXiv listo para usar.\n")
print("Ejemplo 1 → buscar_articulos_arxiv('large language models AND cat:cs.CL')")
print("Ejemplo 2 → generar_respuesta('Resume los avances recientes sobre transformers en arXiv.')")



✅ Agente arXiv listo para usar.

Ejemplo 1 → buscar_articulos_arxiv('large language models AND cat:cs.CL')
Ejemplo 2 → generar_respuesta('Resume los avances recientes sobre transformers en arXiv.')


In [13]:
buscar_articulos_arxiv("large language models AND cat:cs.CL")


🔍 Buscando en arXiv: large language models AND cat:cs.CL
🧾 ID: 2509.16189v2
Título: Latent learning: episodic memory complements parametric learning by enabling flexible reuse of experiences
Autores: Andrew Kyle Lampinen, Martin Engelcke, Yuxuan Li, Arslan Chaudhry, James L. McClelland
Publicado: 2025-09-19
URL: http://arxiv.org/abs/2509.16189v2
Resumen: When do machine learning systems fail to generalize, and what mechanisms could improve their generalization? Here, we draw inspiration from cognitive science to argue that one weakness of parametric machine learning systems is their failure to exhibit latent learning -- learning information that is not relevant to the task at hand, but that might be useful in a future task. We show how this perspective links failures ranging from the reversal curse in language modeling to new findings on agent-based navigation. We then highlight how cognitive science points to episodic memory as a potential part of the solution to these issues. Corres

In [11]:
generar_respuesta("Resume los artículos recientes sobre aprendizaje por refuerzo en arXiv.")



🗣️ Enviando solicitud al modelo...
Prompt: Resume los artículos recientes sobre aprendizaje por refuerzo en arXiv.


🧩 Respuesta del modelo:

Resume los artículos recientes sobre aprendizaje por refuerzo en arXiv. Arxiv es una base de datos con publicaciones de investigación en física, matemáticas, ciencias de la computación, ingeniería y ciencias sociales. Aquí hay un resumen de algunos artículos recientes sobre aprendizaje por refuerzo (RL) que se han publicado en arXiv:

1. "Learning to Navigate in Complex Environments with Reinforcement Learning" - Este trabajo presenta un algoritmo de RL para la navegación en entornos complejos, demostrando su eficacia en tareas de simulación.

2. "Exploration in Reinforcement Learning: A Survey" - Este artículo revisa las técnicas de exploración en RL, proporcionando una visión general de los métodos existentes y sus desafíos.

3. "Reinforcement Learning for Autonomous Driving: A Comprehensive Review" - Se realiza una revisión exhaustiva del uso

'Resume los artículos recientes sobre aprendizaje por refuerzo en arXiv. Arxiv es una base de datos con publicaciones de investigación en física, matemáticas, ciencias de la computación, ingeniería y ciencias sociales. Aquí hay un resumen de algunos artículos recientes sobre aprendizaje por refuerzo (RL) que se han publicado en arXiv:\n\n1. "Learning to Navigate in Complex Environments with Reinforcement Learning" - Este trabajo presenta un algoritmo de RL para la navegación en entornos complejos, demostrando su eficacia en tareas de simulación.\n\n2. "Exploration in Reinforcement Learning: A Survey" - Este artículo revisa las técnicas de exploración en RL, proporcionando una visión general de los métodos existentes y sus desafíos.\n\n3. "Reinforcement Learning for Autonomous Driving: A Comprehensive Review" - Se realiza una revisión exhaustiva del uso de RL en el control automático de vehículos, destacando avances y desafíos.\n\n4. "Deep Reinforcement Learning for Multi-Agent Systems: